In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv('steam_reviews.csv')
# read the file it takes a while

In [3]:
pd.set_option('display.max_columns', None)
df.head()

,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,0,292030,The Witcher 3: Wild Hunt,85185598,schinese,不玩此生遗憾，RPG游戏里的天花板，太吸引人了,1611381629,1611381629,True,0,0,0.0,0,True,False,False,76561199095369542,6,2,1909.0,1448.0,1909.0,1.611343e+09
1,1,292030,The Witcher 3: Wild Hunt,85185250,schinese,拔DIAO无情打桩机--杰洛特!!!,1611381030,1611381030,True,0,0,0.0,0,True,False,False,76561198949504115,30,10,2764.0,2743.0,2674.0,1.611386e+09
2,2,292030,The Witcher 3: Wild Hunt,85185111,schinese,巫师3NB,1611380800,1611380800,True,0,0,0.0,0,True,False,False,76561199090098988,5,1,1061.0,1061.0,1060.0,1.611384e+09
3,3,292030,The Witcher 3: Wild Hunt,85184605,english,"One of the best RPG's of all time, worthy of a...",1611379970,1611379970,True,0,0,0.0,0,True,False,False,76561199054755373,5,3,5587.0,3200.0,5524.0,1.611384e+09
4,4,292030,The Witcher 3: Wild Hunt,85184287,schinese,大作,1611379427,1611379427,True,0,0,0.0,0,True,False,False,76561199028326951,7,4,217.0,42.0,217.0,1.610788e+09


In [4]:
# Keep only necessary columns
df_processed = df[["app_name", "review", "language", "votes_helpful"]]

# Remove missing values
df_processed = df_processed.dropna(subset=["review", "app_name"])

# Keep only English reviews
df_processed = df_processed[df_processed["language"] == "english"]

# Remove very short reviews (noise)
df_processed = df_processed[df_processed["review"].str.len() > 50]

print(df_processed.shape)

(5202331, 4)


In [5]:
def clean_text(text, max_chars=300):
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+", "", text)
    
    # Remove HTML / markdown-like tags
    text = re.sub(r"\[.*?\]", "", text)
    
    # Remove unwanted characters (keep letters, numbers, basic punctuation)
    text = re.sub(r"[^a-z0-9\s\.,!?':-]", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    # Truncate WITHOUT cutting words
    if len(text) > max_chars:
        text = text[:max_chars]
        text = text.rsplit(" ", 1)[0]  # cut at last full word
    
    return text


df_processed["review"] = df_processed["review"].apply(clean_text)

In [6]:
# Sort reviews by helpfulness (descending)
df_processed = df_processed.sort_values(by=["app_name", "votes_helpful"], ascending=[True, False])

In [7]:
grouped_data = []

for app_name, group in df_processed.groupby("app_name"):
    top_reviews = group.head(5)  # take top 5 helpful reviews
    
    # Skip if not enough reviews
    if len(top_reviews) < 5:
        continue
    
    reviews = top_reviews["review"].tolist()
    
    # Combine reviews into one input
    combined_reviews = "\n".join(
        [f"Review {i+1}: {review}" for i, review in enumerate(reviews)]
    )
    
    input_text = f"""Summarize the following Steam game reviews for "{app_name}" in 2-3 sentences:\n\n{combined_reviews}"""
    
    grouped_data.append({
        "app_name": app_name,
        "input": input_text
    })

# Convert to DataFrame
df_processed = pd.DataFrame(grouped_data)

print(df_processed.shape)

(315, 2)


In [8]:
df_processed.to_csv("processed_reviews.csv", index=False)
df_processed_csv = pd.read_csv('processed_reviews.csv')
pd.set_option('display.max_rows', None)

df_processed_csv

,app_name,input
0,20XX,Summarize the following Steam game reviews for...
1,A Hat in Time,Summarize the following Steam game reviews for...
2,A Short Hike,Summarize the following Steam game reviews for...
3,A Way Out,Summarize the following Steam game reviews for...
4,ARK: Survival Evolved,Summarize the following Steam game reviews for...
5,ATLAS,Summarize the following Steam game reviews for...
6,Age of Empires II (2013),Summarize the following Steam game reviews for...
7,Age of Empires: Definitive Edition,Summarize the following Steam game reviews for...
8,American Truck Simulator,Summarize the following Steam game reviews for...
9,Among Us,Summarize the following Steam game reviews for...
